# Heatmaps: transcript isoform and ORF expression

One objective: see whether samples of the same group cluster together, using the normalized expression matrices the pipeline wrote in S4. Each heatmap is drawn with all samples, and again without the samples named in `exclude`.

In [ ]:
source("src/heatmap.R")
source("src/paths.R")
loc <- run_locations()        # every folder of the run, from paths.json
comparison <- loc$comparison
exclude <- c("420_20190402")   # samples to leave out in the second drawing; character(0) skips it
groups <- read_groups(file.path(loc$leafcutter, paste0(comparison, ".lr_leafcutter.groups_file.txt")))
groups

**Explanation.** The line that starts with `loc` reads `paths.json`, the one place where the run's results folder, the comparison name and the dataset name are set. The other cells use short names such as `loc$leafcutter`, so to look at another run you change `paths.json` and nothing in the notebook. The groups file lists each sample and its group. It tells the heatmap which columns to use and how to colour the group bar. `exclude` names the samples to leave out in the second drawing of each heatmap; make it `character(0)` to skip that drawing.

In [ ]:
mat <- read_expression(file.path(loc$de_transcript, paste0(comparison, "_DTE_edgeR_normalized_CPM_matrix.txt")),
                       groups, id_col = "isoform_id", label_col = "transcript_name")
orf <- read_expression(file.path(loc$de_orf, paste0(comparison, "_DE_ORF_edgeR_normalized_CPM_matrix.txt")),
                       groups, id_col = "orf_all_isoform_id", label_col = "orf_base_id")
rbind(isoforms = detected_per_sample(mat), orfs = detected_per_sample(orf))

**Explanation.** For each sample, the number of transcript isoforms (first row) and ORFs (second row) with any expression, from the lowest sample to the highest. A sample far below the others has little coverage, and that alone can make its column look different on a heatmap.

In [ ]:
show_heatmap(expression_heatmap(mat, groups, title = "Transcript isoforms, all samples"), "isoform_heatmap")

**Explanation.** Each row is a transcript isoform, each column a sample. Values are log2(CPM + 1) with every row centred and scaled, so colour shows whether a sample is above (red) or below (blue) that isoform's average, not how large the isoform is. Rows and columns are grouped by similarity (hierarchical clustering). CPM is counts per million reads.

In [ ]:
show_heatmap(expression_heatmap(orf, groups, title = "ORFs, all samples"), "orf_heatmap")

**Explanation.** The same for open reading frames (ORFs), the predicted protein-coding stretches. Each row is an ORF named by its first transcript. Every row has its name, and the figure is as tall as the rows need, with a small font. For this many rows the picture in the notebook is very long; the PDF in `figures/` is the same picture and can be zoomed to read each name.

In [ ]:
if (length(exclude) > 0) {
  g2 <- drop_samples(groups, exclude)
  label <- paste(exclude, collapse = ", ")
  show_heatmap(expression_heatmap(mat[, g2$sample, drop = FALSE], g2, title = paste0("Transcript isoforms, without ", label)), "isoform_heatmap_without")
  show_heatmap(expression_heatmap(orf[, g2$sample, drop = FALSE], g2, title = paste0("ORFs, without ", label)), "orf_heatmap_without")
} else {
  message("exclude is empty: no second drawing")
}

**Explanation.** The same two heatmaps without the samples named in `exclude`. Compare each with its all-samples drawing above: if the groups separate more clearly, or the odd column stops driving the clustering, the excluded sample was influencing the picture. Leaving a sample out after seeing that it looks different needs a reason, so look at the detected counts first and report both versions.